In [2]:
import numpy as np
import pandas as pd

# ─── Tournament Weights ────────────────────────────────────────────────────
def get_tournament_weight(tournament: str) -> float:
    t = str(tournament).lower().strip()
    if "fifa world cup" in t or t == "world cup":
        return 2.00
    if "afc championship" in t or "afc asian cup" in t or "asian cup" in t:
        return 1.80
    if "friendly" in t:
        return 0.96
    return 1.20


# ─── Loss Constants ────────────────────────────────────────────────────────
EXACT_PENALTY            = 0.30
OUTCOME_PENALTY          = 0.25
GD_PENALTY               = 0.15
WRONG_OUTCOME_MULTIPLIER = 1.50
NONLINEAR_POWER          = 1.50


def _outcome(a: int, b: int) -> int:
    if a > b:  return  1
    if a < b:  return -1
    return 0


def official_match_loss(
    y_team_true, y_opp_true,
    y_team_pred, y_opp_pred,
):
    """Per-match loss sesuai formula resmi kompetisi."""
    y_team_true = np.asarray(y_team_true).astype(int)
    y_opp_true  = np.asarray(y_opp_true).astype(int)
    y_team_pred = np.asarray(y_team_pred).astype(int)
    y_opp_pred  = np.asarray(y_opp_pred).astype(int)

    base_mae = (
        np.abs(y_team_true - y_team_pred) +
        np.abs(y_opp_true  - y_opp_pred)
    ) / 2.0

    exact_hit    = (y_team_true == y_team_pred) & (y_opp_true == y_opp_pred)
    true_outcome = np.vectorize(_outcome)(y_team_true, y_opp_true)
    pred_outcome = np.vectorize(_outcome)(y_team_pred, y_opp_pred)
    outcome_hit  = (true_outcome == pred_outcome)
    gd_hit       = ((y_team_true - y_opp_true) == (y_team_pred - y_opp_pred))

    penalty = (
        (~exact_hit).astype(float)   * EXACT_PENALTY +
        (~outcome_hit).astype(float) * OUTCOME_PENALTY +
        (~gd_hit).astype(float)      * GD_PENALTY
    )

    raw_error = base_mae + penalty
    raw_error = np.where(outcome_hit, raw_error, raw_error * WRONG_OUTCOME_MULTIPLIER)

    return raw_error ** NONLINEAR_POWER


def awmae_score(
    y_team_true, y_opp_true,
    y_team_pred, y_opp_pred,
    weights=None,
) -> float:
    """Official AW-MAE: tournament-weighted average per-match loss."""
    losses = official_match_loss(y_team_true, y_opp_true, y_team_pred, y_opp_pred)
    if weights is None:
        weights = np.ones(len(losses), dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)
    return float(np.average(losses, weights=weights))

In [6]:
pd.set_option('display.max_colwidth', None)

In [3]:
def evaluate_submission(gt_path, pred_path, round_pred=False):
    gt = pd.read_csv(gt_path)
    pred = pd.read_csv(pred_path)
    
    # cek kolom minimal
    required_cols = {"Id", "team_goals", "opp_goals"}
    if not required_cols.issubset(gt.columns):
        raise ValueError(f"Ground truth harus punya kolom {required_cols}, kolom saat ini: {list(gt.columns)}")
    if not required_cols.issubset(pred.columns):
        raise ValueError(f"Prediction file harus punya kolom {required_cols}, kolom saat ini: {list(pred.columns)}")

    gt_use = gt.copy()
    pred_use = pred.copy()

    # optional: kalau prediksi float dan mau dibulatkan dulu
    if round_pred:
        pred_use["team_goals"] = np.rint(pred_use["team_goals"]).astype(int)
        pred_use["opp_goals"] = np.rint(pred_use["opp_goals"]).astype(int)

    df = gt_use.merge(
        pred_use[["Id", "team_goals", "opp_goals"]],
        on="Id",
        how="inner",
        suffixes=("_true", "_pred")
    )

    if len(df) != len(gt_use):
        print(f"Warning: hasil merge {len(df)} baris, sedangkan GT {len(gt_use)} baris")

    # weights kalau ada kolom tournament
    if "tournament" in gt_use.columns:
        weights_df = gt_use[["Id", "tournament"]].copy()
        df = df.merge(weights_df, on="Id", how="left")
        weights = df["tournament"].apply(get_tournament_weight).values
    else:
        weights = None

    score = awmae_score(
        y_team_true=df["team_goals_true"],
        y_opp_true=df["opp_goals_true"],
        y_team_pred=df["team_goals_pred"],
        y_opp_pred=df["opp_goals_pred"],
        weights=weights
    )

    return score, df

In [7]:
submission_files = [
    "outputs\exp01_shared_feature_catboost_metric_aware\submissions\submission_exp01_best.csv",
    "outputs\exp02_1_history_engine_repair_fair_validation\submissions\submission_exp02_1_best.csv",
    "best_so_far.csv",
    "outputs\exp03_probabilistic_score_modeling\submissions\submission_exp03_best_safe.csv",
    "outputs\exp03_probabilistic_score_modeling\submissions\submission_exp03_best_research.csv",
    "outputs\submission_exp03_best_safe.csv",
    "outputs\submission_exp03_best_research.csv",

]

results = []

for file in submission_files:
    score, _ = evaluate_submission("data/ground_truth_bersih.csv", file, round_pred=False)
    results.append({
        "file": file,
        "awmae": score
    })

result_df = pd.DataFrame(results).sort_values("awmae", ascending=True).reset_index(drop=True)
display(result_df)

,file,awmae
0,best_so_far.csv,2.904161
1,outputs\exp02_1_history_engine_repair_fair_validation\submissions\submission_exp02_1_best.csv,3.090592
2,outputs\exp03_probabilistic_score_modeling\submissions\submission_exp03_best_safe.csv,3.119564
3,outputs\exp01_shared_feature_catboost_metric_aware\submissions\submission_exp01_best.csv,3.202218
4,outputs\submission_exp03_best_research.csv,3.970699
5,outputs\exp03_probabilistic_score_modeling\submissions\submission_exp03_best_research.csv,4.045411
6,outputs\submission_exp03_best_safe.csv,4.254610
